In [1]:
!pip install sktime

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 46.3 MB/s eta 0:00:00


In [2]:
!pip install sktime[all_extras]

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 31.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.7/147.7 KB 19.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.1/217.1 KB 24.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 KB 22.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 KB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.9/228.9 KB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.4/661.4 KB 39.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 45.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━

In [3]:
from sktime.classification.hybrid import HIVECOTEV1

In [4]:
from sktime.datasets import load_from_tsfile_to_dataframe as load_ts

In [5]:
import numpy as np

In [6]:
from sklearn.model_selection import cross_val_score
from sklearn import metrics
from sklearn.model_selection import KFold

In [7]:
X5_small, y5_small = load_ts("ECGFiveDays_TRAIN.ts")

In [8]:
X5_test, y5_test = load_ts("ECGFiveDays_TEST.ts")
X5_small, y5_small = load_ts("ECGFiveDays_TRAIN.ts")
X200_train, y200_train = load_ts("ECG200_TRAIN.ts")
X5000_train, y5000_train = load_ts("ECG5000_TRAIN.ts")
X200_test, y200_test = load_ts("ECG200_TEST.ts")
X5000_test, y5000_test = load_ts("ECG5000_TEST.ts")

In [ ]:
hc1 = HIVECOTEV1()
hc1.fit(X5_small, np.asarray(list(map(lambda x: int(x), y5_small))))
predictions = hc1.predict(X5_test)
print("PRECISION: ", metrics.precision_score(list(map(lambda x: int(x), y5_test)), predictions))
print("RECALL: ",metrics.recall_score(list(map(lambda x: int(x), y5_test)), predictions))
print("F1: ", metrics.f1_score(list(map(lambda x: int(x), y5_test)), predictions))

In [ ]:
hc2 = HIVECOTEV1()
hc2.fit(X200_train, np.asarray(list(map(lambda x: int(x), y200_train))))
predictions = hc2.predict(X200_test)
print("PRECISION: ", metrics.precision_score(list(map(lambda x: int(x), y200_test)), predictions))
print("RECALL: ",metrics.recall_score(list(map(lambda x: int(x), y200_test)), predictions))
print("F1: ", metrics.f1_score(list(map(lambda x: int(x), y200_test)), predictions))

In [ ]:
hc3 = HIVECOTEV1()
hc3.fit(X5000_train, np.asarray(list(map(lambda x: int(x), y5000_train))))
predictions = hc3.predict(X5000_test)
print("PRECISION: ", metrics.precision_score(list(map(lambda x: int(x), y5000_test)), predictions))
print("RECALL: ",metrics.recall_score(list(map(lambda x: int(x), y5000_test)), predictions))
print("F1: ", metrics.f1_score(list(map(lambda x: int(x), y5000_test)), predictions))

In [ ]:
list(map(lambda x: int(x), y5_small))

[1, 1, 2, 2, 1, 2, 2, 1, 2, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 2, 1]

In [9]:
def train_and_kfold_validate(X, y, num_splits = 5):
  kf = KFold(n_splits=num_splits)
  for i, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"Fold {i}:")
    # print(f"  Train: index={X.loc[train_index]}")
    hc1 = HIVECOTEV1()
    hc1.fit(X.loc[train_index], y[train_index])
    predictions = hc1.predict(X.loc[test_index])
    print("PRECISION: ", metrics.precision_score(y[test_index], predictions))
    print("RECALL: ",metrics.recall_score(y[test_index], predictions))
    print("F1: ", metrics.f1_score(y[test_index], predictions))

In [10]:
import torch
import torch.nn as nn

In [23]:
class TimeSeriesCnnModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
        )
        self.linear = nn.Sequential(
            nn.Linear(256*17, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 2),
            #nn.Softmax()
            )
        
    def forward(self, xb):
        data = self.conv(xb)
        data = torch.flatten(data, start_dim=1)
        return self.linear(data)

In [24]:
tsc = TimeSeriesCnnModel()

In [12]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [13]:
def map_to_scores(x):
  if x == 1:
    return torch.tensor([1., 0.])
  if x == 2:
    return torch.tensor([0., 1.])

In [14]:
class TSCDataset(Dataset):
    def __init__(self, X, y):
        self.x = X
        self.y = y
    
    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return torch.tensor([self.x.iloc[i][0]]).float(), map_to_scores(float(self.y[i]))

In [ ]:
x0_ = torch.tensor(X5_small.iloc[0][0])[None, :].float()
y0_ = float(y5_small[0])

In [20]:
dataset_train = TSCDataset(X5_small, y5_small)
dataset_test = TSCDataset(X5_test, y5_test)

In [21]:
batch_size = 4
train_dataloader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(dataset_test, batch_size=batch_size, shuffle=False)

In [18]:
import tqdm
def train_loop(model, epochs, optim, loss_fn, dataloader_train, dataloader_test, n_batches):
    for epoch in range(epochs):
        for i, (X, y) in tqdm.notebook.tqdm(enumerate(dataloader_train), total=n_batches):
            #print(X.shape, y)
            pred = model(X)
            #print(pred)
            loss = loss_fn(pred, y)
            optim.zero_grad()
            loss.backward()
            optim.step()
    
            if i % (n_batches//3) == 0:
                l= loss.item()
                print(f"train loss: {l:>5f}")
                
        #evaluate(model, dataloader_test)
        model.train()

In [25]:
optim = torch.optim.Adam(tsc.parameters(), lr=0.001)
loss_fn = torch.nn.CrossEntropyLoss()
train_loop(tsc, 20, optim, loss_fn, train_dataloader, None, len(dataset_train)//batch_size)

  0%|          | 0/5 [00:00<?, ?it/s]

train loss: 0.693185
train loss: 0.690682
train loss: 0.692850
train loss: 0.693113
train loss: 0.708984
train loss: 0.519846


  0%|          | 0/5 [00:00<?, ?it/s]

train loss: 0.791685
train loss: 1.261168
train loss: 0.569041
train loss: 0.538937
train loss: 0.678360
train loss: 0.575270


  0%|          | 0/5 [00:00<?, ?it/s]

...

train loss: 0.237693
train loss: 0.309221
train loss: 0.187521
train loss: 0.187262
train loss: 0.495241
train loss: 0.333315


  0%|          | 0/5 [00:00<?, ?it/s]

train loss: 0.354968
train loss: 0.203338
train loss: 0.388412
train loss: 0.068246
train loss: 0.375295
train loss: 0.407646


  0%|          | 0/5 [00:00<?, ?it/s]

train loss: 0.156722
train loss: 0.145329
train loss: 0.127732
train loss: 0.121180
train loss: 0.032744
train loss: 0.057962


In [ ]:
#train_and_kfold_validate(X5_small, np.asarray(list(map(lambda x: int(x), y5_small))), num_splits = 3)

In [ ]:
train_and_kfold_validate(X200_train, y200_train)

In [ ]:
train_and_kfold_validate(X5000_train, y5000_train)

In [ ]:
from datetime import datetime

import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_predict
from sklearn.utils import check_random_state

from sktime.classification.base import BaseClassifier
from sktime.classification.dictionary_based import ContractableBOSS
from sktime.classification.interval_based import (
    RandomIntervalSpectralEnsemble,
    TimeSeriesForestClassifier,
)
from sktime.classification.shapelet_based import ShapeletTransformClassifier

In [ ]:
class HIVECOTEV1_MY(HIVECOTEV1):
    _tags = {
        "capability:multithreading": True,
        "classifier_type": "hybrid",
    }

    def __init__(
        self,
        stc_params=None,
        tsf_params=None,
        rise_params=None,
        cboss_params=None,
        verbose=0,
        n_jobs=1,
        random_state=None,
    ):
        self.stc_params = stc_params
        self.tsf_params = tsf_params
        self.rise_params = rise_params
        self.cboss_params = cboss_params

        self.verbose = verbose
        self.n_jobs = n_jobs
        self.random_state = random_state

        self.stc_weight_ = 0
        self.tsf_weight_ = 0
        self.rise_weight_ = 0
        self.cboss_weight_ = 0

        self.cnn_weight_ = 0

        self._stc_params = stc_params
        self._tsf_params = tsf_params
        self._rise_params = rise_params
        self._cboss_params = cboss_params
        self._stc = None
        self._tsf = None
        self._rise = None
        self._cboss = None

        self.cnn = None

        super(HIVECOTEV1, self).__init__()

    def _fit(self, X, y):
        """Fit HIVE-COTE 1.0 to training data.
        Parameters
        ----------
        X : 3D np.array of shape = [n_instances, n_dimensions, series_length]
            The training data.
        y : array-like, shape = [n_instances]
            The class labels.
        Returns
        -------
        self :
            Reference to self.
        Notes
        -----
        Changes state by creating a fitted model that updates attributes
        ending in "_" and sets is_fitted flag to True.
        """
        # Default values from HC1 paper
        if self.stc_params is None:
            self._stc_params = {"transform_limit_in_minutes": 120}
        if self.tsf_params is None:
            self._tsf_params = {"n_estimators": 500}
        if self.rise_params is None:
            self._rise_params = {"n_estimators": 500}
        if self.cboss_params is None:
            self._cboss_params = {}

        # Cross-validation size for TSF and RISE
        cv_size = 10
        _, counts = np.unique(y, return_counts=True)
        min_class = np.min(counts)
        if min_class < cv_size:
            cv_size = min_class

        # Build STC
        self._stc = ShapeletTransformClassifier(
            **self._stc_params,
            save_transformed_data=True,
            random_state=self.random_state,
            n_jobs=self._threads_to_use,
        )
        self._stc.fit(X, y)

        if self.verbose > 0:
            print("STC ", datetime.now().strftime("%H:%M:%S %d/%m/%Y"))  # noqa

        # Find STC weight using train set estimate
        train_probs = self._stc._get_train_probs(X, y)
        train_preds = self._stc.classes_[np.argmax(train_probs, axis=1)]
        self.stc_weight_ = accuracy_score(y, train_preds) ** 4

        if self.verbose > 0:
            print(  # noqa
                "STC train estimate ",
                datetime.now().strftime("%H:%M:%S %d/%m/%Y"),
            )
            print("STC weight = " + str(self.stc_weight_))  # noqa

        # Build TSF
        self._tsf = TimeSeriesForestClassifier(
            **self._tsf_params,
            random_state=self.random_state,
            n_jobs=self._threads_to_use,
        )
        self._tsf.fit(X, y)

        if self.verbose > 0:
            print("TSF ", datetime.now().strftime("%H:%M:%S %d/%m/%Y"))  # noqa

        # Find TSF weight using train set estimate found through CV
        train_preds = cross_val_predict(
            TimeSeriesForestClassifier(
                **self._tsf_params, random_state=self.random_state
            ),
            X=X,
            y=y,
            cv=cv_size,
            n_jobs=self._threads_to_use,
        )
        self.tsf_weight_ = accuracy_score(y, train_preds) ** 4

        if self.verbose > 0:
            print(  # noqa
                "TSF train estimate ",
                datetime.now().strftime("%H:%M:%S %d/%m/%Y"),
            )
            print("TSF weight = " + str(self.tsf_weight_))  # noqa

        # Build RISE
        self._rise = RandomIntervalSpectralEnsemble(
            **self._rise_params,
            random_state=self.random_state,
            n_jobs=self._threads_to_use,
        )
        self._rise.fit(X, y)

        if self.verbose > 0:
            print("RISE ", datetime.now().strftime("%H:%M:%S %d/%m/%Y"))  # noqa

        # Find RISE weight using train set estimate found through CV
        train_preds = cross_val_predict(
            RandomIntervalSpectralEnsemble(
                **self._rise_params,
                random_state=self.random_state,
            ),
            X=X,
            y=y,
            cv=cv_size,
            n_jobs=self._threads_to_use,
        )
        self.rise_weight_ = accuracy_score(y, train_preds) ** 4

        if self.verbose > 0:
            print(  # noqa
                "RISE train estimate ",
                datetime.now().strftime("%H:%M:%S %d/%m/%Y"),
            )
            print("RISE weight = " + str(self.rise_weight_))  # noqa

        # Build cBOSS
        self._cboss = ContractableBOSS(
            **self._cboss_params,
            random_state=self.random_state,
            n_jobs=self._threads_to_use,
        )
        self._cboss.fit(X, y)

        # Find cBOSS weight using train set estimate
        train_probs = self._cboss._get_train_probs(X, y)
        train_preds = self._cboss.classes_[np.argmax(train_probs, axis=1)]
        self.cboss_weight_ = accuracy_score(y, train_preds) ** 4

        if self.verbose > 0:
            print(  # noqa
                "cBOSS (estimate included)",
                datetime.now().strftime("%H:%M:%S %d/%m/%Y"),
            )
            print("cBOSS weight = " + str(self.cboss_weight_))  # noqa

        return self

    def train_add_nn(self, model, epochs, optim, loss_fn, X, y, n_batches):
      
        for epoch in range(epochs):
          for i, (X, y) in tqdm.notebook.tqdm(enumerate(dataloader_train), total=n_batches):
            pred = model(X)
            loss = loss_fn(pred, map_to_scores(y))
            optim.zero_grad()
            loss.backward()
            optim.step()
    
            if i % (n_batches//3) == 0:
                l= loss.item()
                print(f"train loss: {l:>5f}")
                
        self.cnn = model

        # train_preds = self.cnn(X)
        # self.cnn_weight_ = accuracy_score(y, train_preds) ** 4

        # print("cBOSS weight = " + str(self.cnn_weight_))  # noqa


    def _predict(self, X) -> np.ndarray:
        """Predicts labels for sequences in X.
        Parameters
        ----------
        X : 3D np.array of shape = [n_instances, n_dimensions, series_length]
            The data to make predictions for.
        Returns
        -------
        y : array-like, shape = [n_instances]
            Predicted class labels.
        """
        rng = check_random_state(self.random_state)
        return np.array(
            [
                self.classes_[int(rng.choice(np.flatnonzero(prob == prob.max())))]
                for prob in self.predict_proba(X)
            ]
        )

    def _predict_proba(self, X) -> np.ndarray:
        """Predicts labels probabilities for sequences in X.
        Parameters
        ----------
        X : 3D np.array of shape = [n_instances, n_dimensions, series_length]
            The data to make predict probabilities for.
        Returns
        -------
        y : array-like, shape = [n_instances, n_classes_]
            Predicted probabilities using the ordering in classes_.
        """
        dists = np.zeros((X.shape[0], self.n_classes_))

        # Call predict proba on each classifier, multiply the probabilities by the
        # classifiers weight then add them to the current HC1 probabilities
        dists = np.add(
            dists,
            self._stc.predict_proba(X) * (np.ones(self.n_classes_) * self.stc_weight_),
        )
        dists = np.add(
            dists,
            self._tsf.predict_proba(X) * (np.ones(self.n_classes_) * self.tsf_weight_),
        )
        dists = np.add(
            dists,
            self._rise.predict_proba(X)
            * (np.ones(self.n_classes_) * self.rise_weight_),
        )
        dists = np.add(
            dists,
            self._cboss.predict_proba(X)
            * (np.ones(self.n_classes_) * self.cboss_weight_),
        )
        dists = np.add(
            dists,
            self.cnn(X)
            * (np.ones(self.n_classes_) * self.cnn_weight_),
        )

        # Make each instances probability array sum to 1 and return
        return dists / dists.sum(axis=1, keepdims=True)

    @classmethod
    def get_test_params(cls, parameter_set="default"):
        """Return testing parameter settings for the estimator.
        Parameters
        ----------
        parameter_set : str, default="default"
            Name of the set of test parameters to return, for use in tests. If no
            special parameters are defined for a value, will return `"default"` set.
            For classifiers, a "default" set of parameters should be provided for
            general testing, and a "results_comparison" set for comparing against
            previously recorded results if the general set does not produce suitable
            probabilities to compare against.
        Returns
        -------
        params : dict or list of dict, default={}
            Parameters to create testing instances of the class.
            Each dict are parameters to construct an "interesting" test instance, i.e.,
            `MyClass(**params)` or `MyClass(**params[i])` creates a valid test instance.
            `create_test_instance` uses the first (or only) dictionary in `params`.
        """
        from sklearn.ensemble import RandomForestClassifier

        if parameter_set == "results_comparison":
            return {
                "stc_params": {
                    "estimator": RandomForestClassifier(n_estimators=3),
                    "n_shapelet_samples": 50,
                    "max_shapelets": 5,
                    "batch_size": 10,
                },
                "tsf_params": {"n_estimators": 3},
                "rise_params": {"n_estimators": 3},
                "cboss_params": {"n_parameter_samples": 5, "max_ensemble_size": 3},
            }
        else:
            return {
                "stc_params": {
                    "estimator": RandomForestClassifier(n_estimators=1),
                    "n_shapelet_samples": 5,
                    "max_shapelets": 5,
                    "batch_size": 5,
                },
                "tsf_params": {"n_estimators": 1},
                "rise_params": {"n_estimators": 1},
                "cboss_params": {"n_parameter_samples": 1, "max_ensemble_size": 1},
            }

In [ ]:
tsc = TimeSeriesCnnModel()

In [ ]:
hc4 = HIVECOTEV1_MY()

In [ ]:
hc4.fit(X5_small, np.asarray(list(map(lambda x: int(x), y5_small))))

HIVECOTEV1_MY()

In [ ]:
hc4.cnn = tsc

In [ ]:
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score

In [ ]:
def get_f1_acc_recall(pred, target):
    print(f"RECALL SCORE: {recall_score(target, pred, average='macro')}")
    print(f"ACCURACY SCORE: {balanced_accuracy_score(target, pred)}")
    print(f"F1 SCORE: {f1_score(target, pred, average='macro')}")  

In [ ]:
def evaluate(model, dataloader_test):
    lst = []
    targets = []
    model.eval()
    with torch.no_grad():
        for X, y in dataloader_test:
            pred = model(X)
            lst.append([(torch.argmax(pred, dim=0) + 1).numpy()])
            targets.append(y.numpy())
    lst = np.concatenate(lst, axis=0)
    targets = np.concatenate(targets)
    get_f1_acc_recall(lst, targets)
    return

In [ ]:
evaluate(tsc, test_dataloader)

RECALL SCORE: 0.8738641514320865
ACCURACY SCORE: 0.8738641514320865
F1 SCORE: 0.8726996507172167


In [ ]:
for _, (X, y) in enumerate(train_dataloader):
    pred = hc4.cnn(X)
    print(pred)

tensor([8.1214e-19, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([8.2513e-15, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([1.4058e-17, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([4.4035e-16, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([1.6413e-17, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([1.0944e-16, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([2.3650e-15, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([6.5591e-16, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([2.0212e-17, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([1.1357e-14, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([7.3422e-17, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([1.1751e-16, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([3.2871e-16, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([2.1475e-17, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([4.2632e-17, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([3.6714e-16, 1.0000e+00], grad_fn=<SoftmaxBackward0>)
tensor([4.4336e-16, 1.00

In [ ]:
hc4._cboss.predict_proba(X5_small)

array([[0.88097677, 0.11902323],
       [1.        , 0.        ],
       [0.03316263, 0.96683737],
       [0.1141394 , 0.8858606 ],
       [0.93560455, 0.06439545],
       [0.        , 1.        ],
       [0.05951162, 0.94048838],
       [1.        , 0.        ],
       [0.02146515, 0.97853485],
       [0.93560455, 0.06439545],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.09267425, 0.90732575],
       [0.03316263, 0.96683737],
       [0.01658132, 0.98341868],
       [1.        , 0.        ],
       [0.82634898, 0.17365102],
       [0.97853485, 0.02146515],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.02146515, 0.97853485],
       [1.        , 0.        ]])

In [ ]:
np.ones(hc4.n_classes_)

array([1., 1.])

In [ ]:
hc4.cboss_weight_

1.0

In [ ]:
hc4._cboss.predict_proba(X5_small) * (np.ones(hc4.n_classes_) * hc4.cboss_weight_)

array([[0.88097677, 0.11902323],
       [1.        , 0.        ],
       [0.03316263, 0.96683737],
       [0.1141394 , 0.8858606 ],
       [0.93560455, 0.06439545],
       [0.        , 1.        ],
       [0.05951162, 0.94048838],
       [1.        , 0.        ],
       [0.02146515, 0.97853485],
       [0.93560455, 0.06439545],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.09267425, 0.90732575],
       [0.03316263, 0.96683737],
       [0.01658132, 0.98341868],
       [1.        , 0.        ],
       [0.82634898, 0.17365102],
       [0.97853485, 0.02146515],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.02146515, 0.97853485],
       [1.        , 0.        ]])